# 13 — Comparison of causal completeness across SAE checkpointsRe-evaluate full residual, whole SAE, reconstruction error, sparse features, active energy-matched controls, sign-flip and spatial permutation at the same SD3 site, prompts and seeds. **Feature discovery is recomputed separately for each SAE**, strictly from the discovery split of notebook 09. The confirmatory split is not used for ranking.

In [ ]:
MODE = "pilot"  # pilot / fullSEED = 2601MODEL_NAMES = ["topk_x2_k16","topk_x4_k32","temporal_x4_k32"]if MODE=="full": MODEL_NAMES=["topk_x2_k16","topk_x4_k32","topk_x8_k64","temporal_x4_k32"]EVALUATION_SEEDS = (0,) if MODE=="pilot" else (1,2)MAX_PAIRS_PER_LABEL = 1 if MODE=="pilot" else NoneFORCE_RERUN=False

## 1. Environment and frozen data provenanceThe original notebook 09 discovery split is reused only for feature ranking; pair-level intervention results are computed on distinct evaluation/confirmatory pairs. DataSphere artifacts remain under separate output directories.

In [ ]:
import json,os,sysfrom pathlib import Pathimport pandas as pdimport torchfrom IPython.display import displaydef root():    wd=Path.cwd().resolve()    for p in [wd,*wd.parents,Path("/home/jupyter/project/diffusion_attention_analysis_2")]:        if (p/"src/diffusion_attention_analysis_v2").is_dir():return p    raise FileNotFoundError("Project root unavailable")ROOT=root();os.chdir(ROOT);sys.path.insert(0,str(ROOT/"src"))from diffusion_attention_analysis_v2.config import load_config,resolve_pathfrom diffusion_attention_analysis_v2.sd3_notebook_utils import prepare_datashere_storagefrom diffusion_attention_analysis_v2.sd3_counterfactual_v4 import run_campaignS=prepare_datashere_storage(project_root=ROOT,run_tag="sd3_medium_sae_circuit_150",    container_name="container4",require_container=True,min_free_gb=12.)assert S["free_space_ok"]cfg=load_config(ROOT/"configs/06_causal_analysis/sd3_medium_counterfactual_sae_mediation_v2.yaml")PREV=resolve_path(cfg,"data.output_dir",required=True)manifest=pd.read_csv(PREV/"split_manifest.csv")assert {"discovery","evaluation","confirmatory"}.issubset(set(manifest.split))RUN=ROOT/"outputs/sd3_medium_sae_circuit_150"OUT=RUN/"08_sae_causal_model_comparison"IDX=RUN/"07_sae_fidelity_rescue/checkpoint_index.json"if not IDX.is_file():raise FileNotFoundError("Run notebook 12 before notebook 13; checkpoint index missing")CHECKPOINTS=json.loads(IDX.read_text())ROOTS=[PREV,RUN/"06_causal_completeness_v3",RUN/"06_causal_completeness_v4"]assert torch.cuda.is_available()print("Experiment manifest:",len(manifest),"rows; models:",MODEL_NAMES)

## 2. Per-model causal intervention and downstream tracingFull-window residual replacement is the positive control. For each candidate, discovery takes place on the corresponding new encoder activations: original SAE feature IDs are NOT reused. All control doses are validated to be nonzero. Results and images are stored separately per model.

In [ ]:
results=[]for name in MODEL_NAMES:    if name not in CHECKPOINTS:raise KeyError(f"Checkpoint {name} not found in notebook 12 index")    checkpoint=Path(CHECKPOINTS[name])    assert checkpoint.is_file(),checkpoint    destination=OUT/name    summary,primary=run_campaign(cfg,manifest,output=destination,        reference_roots=ROOTS,checkpoint=checkpoint,discovery_seed=0,        evaluation_seeds=EVALUATION_SEEDS,max_eval_pairs=MAX_PAIRS_PER_LABEL,        force=FORCE_RERUN)    summary.insert(0,"model",name)    results.append(summary)    print(name,primary)all_results=pd.concat(results,ignore_index=True)OUT.mkdir(parents=True,exist_ok=True)all_results.to_csv(OUT/"cross_model_causal_summary.csv",index=False)display(all_results)

## 3. Compare controlled semantic recovery and structural fidelityThis comparison is descriptive until fully independent prompts are tested in notebook 14. The primary effect is the paired semantic K=16 minus an active dose-matched nonsemantic control. Reconstructed whole-state effects are not additive causal shares.

In [ ]:
import matplotlib.pyplot as pltp=all_results.pivot(index="model",columns="arm",values="margin")for key in ["full","whole_sae","error","semantic_16","active_control_16","signflip_16","permutation_16"]:    if key not in p:raise KeyError(key)ax=p[["full","whole_sae","error","semantic_16","active_control_16"]].plot.bar(figsize=(10,5))ax.set_ylabel("Mean target−replacement margin change")ax.axhline(0,color="black",linewidth=.7)plt.tight_layout()plt.savefig(OUT/"causal_margin_by_checkpoint.png",dpi=180)plt.show()quality=all_results.pivot(index="model",columns="arm",values="ssim")report=pd.DataFrame({    "model":p.index,    "semantic_minus_control":(p.semantic_16-p.active_control_16).values,    "semantic_minus_signflip":(p.semantic_16-p.signflip_16).values,    "semantic_minus_permutation":(p.semantic_16-p.permutation_16).values,    "whole_vs_full":(p.whole_sae/p.full.replace(0,float("nan"))).values,    "semantic_ssim":quality.semantic_16.values,})report.to_csv(OUT/"cross_model_diagnostic_differences.csv",index=False)display(report)print("Next: notebook 14 freezes a new disjoint prompt manifest and repeats the comparison.")